# Olist
Личный аналитический проект на открытых данных. Качество витрины, доставка и повторные покупки; коммерческое внедрение не заявляется.

## Данные и подготовка

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=Path.cwd() if (Path.cwd()/"data/olist_orders_dataset.csv").exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from src.analysis import analyse,prepare_orders
names=["olist_orders_dataset.csv","olist_customers_dataset.csv","olist_order_items_dataset.csv","olist_order_reviews_dataset.csv"]
raw=[pd.read_csv(ROOT/"data"/n) for n in names]
summary,tables=analyse(*raw)
print(json.dumps(summary,ensure_ascii=False,indent=2))

{
  "orders": 99441,
  "review_rows": 99224,
  "latest_review_rows": 98673,
  "removed_review_rows": 551,
  "orders_without_items": 775,
  "invalid_delivered_dates": 31,
  "currency": "BRL",
  "grain": "one row per order",
  "delivered_orders": 96478,
  "valid_delivery_reviews": 95801,
  "retention_cutoff": "2018-08",
  "cohort_population": "delivered orders; unique customer; observed purchase month; no proof of full history before extract",
  "weighted_m1_retention_pct": 0.4815851029674815,
  "average_delivered_order_goods_brl": 137.0415857501192,
  "scenario": {
    "baseline_m1": 0.004815851029674815,
    "target_m1": 0.02,
    "mean_eligible_new_customers_per_cohort": 4360.6,
    "assumed_repeat_orders_per_returner": 1,
    "assumed_repeat_aov_brl": 137.0415857501192,
    "illustrative_incremental_gmv_per_cohort_brl": 9073.797475686895,
    "interpretation": "Hypothetical incremental merchandise GMV, not profit, guaranteed revenue or measured intervention effect."
  },
  "limits": 

## Доставка и оценки
Интервалы имеют дробную длительность в сутках. Это наблюдаемая связь, не причинный эффект. Исторический порог 19 дней считается исследовательским.

In [2]:
print(tables["delivery"].to_string(index=False))

delivery_band  orders  mean_score  one_star_share
          0–7   25917    4.419261        0.053093
         7–14   39992    4.312162        0.064913
        14–19   14073    4.157394        0.082427
        19–30   11389    3.748266        0.157872
          30+    4430    2.252596        0.545147


## Регионы
Сравниваем группы не менее 100 валидных отзывов.

In [3]:
print(tables["regions"].query("orders>=100").sort_values("mean_delivery_days",ascending=False).to_string(index=False))

customer_state  orders  mean_delivery_days  mean_score
            AM     144           26.352644    4.243056
            AL     394           24.457127    3.847716
            PA     933           23.641464    3.914255
            MA     712           21.441588    3.832865
            SE     334           21.438862    3.907186
            CE    1273           21.205486    3.944226
            PB     512           20.246941    4.076172
            PI     471           19.494019    3.993631
            RO     242           19.373472    4.165289
            RN     471           19.254033    4.150743
            BA    3229           19.245509    3.929700
            PE    1579           18.371846    4.082964
            MT     879           17.965479    4.145620
            TO     273           17.676524    4.153846
            MS     699           15.626170    4.161660
            ES    1969           15.609603    4.078720
            GO    1946           15.558286    4.103803
          

## Продавцы
Заказ с несколькими продавцами исключён из атрибуции одному продавцу. Минимум 10 отзывов; даты доставки валидируются отдельно.

In [4]:
print(tables["sellers"].head(15).to_string(index=False))

                       seller_id  orders  reviews  mean_score  mean_valid_delivery_days  merchandise_gmv_brl
4342d4b2ba6b161468c63a7e7cfce593      20       19    1.263158                 15.676088              2867.10
b1b3948701c5c72445495bd161b83a4c      18       18    1.722222                 23.403023             24699.19
ffff564a4f9085cd26170f4732393726      20       20    2.100000                 12.496233              1426.30
1ca7077d890b907f89be8c954a02686a     113      112    2.330357                 15.569003             13191.57
40db9e9aa57f7bb151bcda6b0f9bdbb7      12       11    2.363636                 14.775632             23168.00
fa74b2f3287d296e9fbd2cc80f2d1cf1      11       11    2.363636                 30.380074              2459.89
5bc55dbe2f12b6af6d83ed46023e0dc8      14       14    2.500000                 16.260857              2392.50
b5abf4f36adc043117b4fca82c22984c      10       10    2.500000                 21.767320              1197.90
ecccfa2bb93b34a3bf0

## Месячные показатели
Товарный GMV завершённых заказов, без доставки. Валюта BRL.

In [5]:
print(tables["monthly"].to_string(index=False))

order_month  orders  merchandise_gmv_brl
    2016-09       1               134.97
    2016-10     265             40325.11
    2016-12       1                10.90
    2017-01     750            111798.36
    2017-02    1653            234223.40
    2017-03    2546            359198.85
    2017-04    2303            340669.68
    2017-05    3546            489338.25
    2017-06    3135            421923.37
    2017-07    3872            481604.52
    2017-08    4193            554699.70
    2017-09    4150            607399.67
    2017-10    4478            648247.65
    2017-11    7289            987765.37
    2017-12    5513            726033.19
    2018-01    7069            924645.00
    2018-02    6555            826437.13
    2018-03    7003            953356.25
    2018-04    6798            973534.09
    2018-05    6749            977544.69
    2018-06    6099            856077.86
    2018-07    6159            867953.46
    2018-08    6351            838576.64


## Retention
Статус delivered; уникальный покупатель; cutoff до построения когорт. Наблюдаемый ноль отличается от ненаступившего периода.

In [6]:
print(tables["retention_pct"].iloc[:,:7].round(3).to_string())
print("Взвешенный M1, %:",summary["weighted_m1_retention_pct"])

age              0      1      2      3      4      5      6
order_month                                                 
2016-10      100.0  0.000  0.000  0.000  0.000  0.000  0.382
2017-01      100.0  0.279  0.279  0.139  0.418  0.139  0.418
2017-02      100.0  0.184  0.307  0.123  0.430  0.123  0.246
2017-03      100.0  0.439  0.360  0.400  0.360  0.160  0.160
2017-04      100.0  0.621  0.222  0.177  0.266  0.266  0.355
2017-05      100.0  0.464  0.464  0.290  0.290  0.319  0.406
2017-06      100.0  0.494  0.395  0.428  0.296  0.395  0.362
2017-07      100.0  0.533  0.346  0.240  0.293  0.213  0.320
2017-08      100.0  0.690  0.345  0.271  0.345  0.518  0.296
2017-09      100.0  0.699  0.549  0.275  0.450  0.225  0.225
2017-10      100.0  0.716  0.254  0.092  0.231  0.208  0.208
2017-11      100.0  0.567  0.368  0.170  0.170  0.184  0.113
2017-12      100.0  0.206  0.281  0.337  0.262  0.206  0.169
2018-01      100.0  0.336  0.365  0.292  0.292  0.161  0.175
2018-02      100.0  0.35

## Сценарий роста повторных покупок
Число новых покупателей, средний товарный чек и одна повторная покупка — явные допущения. Это не прибыль и не измеренный эффект внедрения.

In [7]:
print(json.dumps(summary["scenario"],ensure_ascii=False,indent=2))

{
  "baseline_m1": 0.004815851029674815,
  "target_m1": 0.02,
  "mean_eligible_new_customers_per_cohort": 4360.6,
  "assumed_repeat_orders_per_returner": 1,
  "assumed_repeat_aov_brl": 137.0415857501192,
  "illustrative_incremental_gmv_per_cohort_brl": 9073.797475686895,
  "interpretation": "Hypothetical incremental merchandise GMV, not profit, guaranteed revenue or measured intervention effect."
}


## Обзор для бизнеса
Выводы находятся в reports/EXECUTIVE_SUMMARY.md. Дашборд: docs/index.html; изображение в README. Полнота месяца и история до выгрузки не доказаны одним CSV.

In [8]:
print("Пересчёт HTML, PNG, CSV и summary.json: python -m scripts.build_report")

Пересчёт HTML, PNG, CSV и summary.json: python -m scripts.build_report
